In [4]:
from tlsmate.tlsmate import TlsMate
from tlsmate import tls, msg

tlsmate = TlsMate()

# To execute a successful TLS handshake we must tell `tlsmate` where to find
# the trust store(s) containing the root certificates. Those trust stores are
# files which contain a set of certificates in PEM format. This example uses
# the trust store from a typical Ubuntu system. For demonstation purposes we
# set the trust store hard coded in the application. Later we will see how we
# can define such a common setting via an ini-file or via an environment variable.
tlsmate.trust_store.set_ca_files(["/etc/ssl/certs/ca-certificates.crt"])

# Let's use a default client profile which has a high probability to successfully
# interoperate with a typical web server. Additionally we will exclude TLS1.3
# here explicitly to demonstrate how to implement a typical TLS1.0 - TLS1.2
# handshake scenario.
tlsmate.client.set_profile(tls.Profile.INTEROPERABILITY)
tlsmate.client.profile.versions = [
    tls.Version.TLS10, tls.Version.TLS11, tls.Version.TLS12
]

# Now open a TLS connection and execute a typical TLS handshake. Print the
# cipher suite selected by the server.
with tlsmate.client.create_connection("google.com") as conn:
    conn.send(msg.ClientHello)
    server_hello = conn.wait(msg.ServerHello)
    print(f"Cipher suite selected by the server: {server_hello.cipher_suite}")
    conn.wait(msg.Certificate, optional=True)
    conn.wait(msg.ServerKeyExchange, optional=True)
    conn.wait(msg.ServerHelloDone)
    conn.send(msg.ClientKeyExchange)
    conn.send(msg.ChangeCipherSpec)
    conn.send(msg.Finished)
    conn.wait(msg.ChangeCipherSpec)
    conn.wait(msg.Finished)
    print("Handshake finished")

Cipher suite selected by the server: TLS_ECDHE_ECDSA_WITH_AES_128_GCM_SHA256
Handshake finished


In [ ]:
import ssl
import tempfile
from cryptography import x509
from cryptography.x509.oid import NameOID
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import rsa
import datetime

def generate_mock_credentials_pem():
    """Generates a temporary self-signed certificate and key in PEM format."""
    key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
    subject = issuer = x509.Name([
        x509.NameAttribute(NameOID.COUNTRY_NAME, "US"),
        x509.NameAttribute(NameOID.ORGANIZATION_NAME, "Standard TLS Lab"),
        x509.NameAttribute(NameOID.COMMON_NAME, "localhost"),
    ])
    cert = x509.CertificateBuilder().subject_name(subject).issuer_name(issuer).public_key(
        key.public_key()
    ).serial_number(x509.random_serial_number()).not_valid_before(
        datetime.datetime.now(datetime.timezone.utc)
    ).not_valid_after(
        datetime.datetime.now(datetime.timezone.utc) + datetime.timedelta(days=1)
    ).sign(key, hashes.SHA256())
    
    from cryptography.hazmat.primitives import serialization
    cert_pem = cert.public_bytes(serialization.Encoding.PEM)
    key_pem = key.private_bytes(
        encoding=serialization.Encoding.PEM,
        format=serialization.PrivateFormat.TraditionalOpenSSL,
        encryption_algorithm=serialization.NoEncryption()
    )
    return cert_pem, key_pem

def simulate_standard_endpoints():
    # 1. Initialize Native Python SSL Contexts for TLS 1.3
    server_ctx = ssl.SSLContext(ssl.PROTOCOL_TLS_SERVER)
    server_ctx.minimum_version = ssl.TLSVersion.TLSv1_3
    server_ctx.maximum_version = ssl.TLSVersion.TLSv1_3

    client_ctx = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
    client_ctx.minimum_version = ssl.TLSVersion.TLSv1_3
    client_ctx.maximum_version = ssl.TLSVersion.TLSv1_3
    
    # Configure the curves safely using native Python methods
    try:
        server_ctx.set_ecdh_curve("X25519")
        client_ctx.set_ecdh_curve("X25519")
    except Exception as e:
        print(f"Curve configuration warning: {e}")

    # Generate mock credentials and write them to a secure temporary file
    cert_pem, key_pem = generate_mock_credentials_pem()
    
    # Using NamedTemporaryFile to pass a valid file path to load_cert_chain
    with tempfile.NamedTemporaryFile(delete=True) as cert_file:
        cert_file.write(cert_pem + key_pem)
        cert_file.flush()  # Force bytes to write out to disk
        server_ctx.load_cert_chain(certfile=cert_file.name)
    
        # Bypass certificate verification on client side for memory simulation
        client_ctx.check_hostname = False
        client_ctx.verify_mode = ssl.CERT_NONE

        # 2. Establish Pure In-Memory MemoryBIO buffers
        client_incoming = ssl.MemoryBIO()
        client_outgoing = ssl.MemoryBIO()
        
        server_incoming = ssl.MemoryBIO()
        server_outgoing = ssl.MemoryBIO()

        # Create the internal TLS state objects
        client_obj = client_ctx.wrap_bio(client_incoming, client_outgoing, server_side=False)
        server_obj = server_ctx.wrap_bio(server_incoming, server_outgoing, server_side=True)

        print("--- Beginning In-Memory Native TLS 1.3 Classical Handshake ---")
        
        # 3. Simulate the state machine handshake loop
        iterations = 0
        client_done = False
        server_done = False

        while iterations < 30:  # Safety cap to prevent infinite loops
            iterations += 1
            
            # --- Client Step ---
            if not client_done:
                try:
                    client_obj.do_handshake()
                    client_done = True
                except (ssl.SSLWantReadError, ssl.SSLWantWriteError):
                    pass

            # Flush data from Client outgoing buffer to Server incoming buffer
            if client_outgoing.pending > 0:
                client_data = client_outgoing.read()
                print(f"[Client -> Server] Transferring {len(client_data)} bytes of handshake data...")
                server_incoming.write(client_data)

            # --- Server Step ---
            if not server_done:
                try:
                    server_obj.do_handshake()
                    server_done = True
                except (ssl.SSLWantReadError, ssl.SSLWantWriteError):
                    pass

            # Flush data from Server outgoing buffer to Client incoming buffer
            if server_outgoing.pending > 0:
                server_data = server_outgoing.read()
                print(f"[Server -> Client] Transferring {len(server_data)} bytes of handshake data...")
                client_incoming.write(server_data)

            # Break when both endpoint engines declare the handshake completed
            if client_done and server_done:
                print("\n[Success] Handshake finished completely in-memory!")
                break
                
        # 4. Verify selected group protocol info
        cipher_info = client_obj.cipher()
        if cipher_info:
            print(f"Negotiated Cipher Suite details: {cipher_info}")

if __name__ == "__main__":
    simulate_standard_endpoints()


TypeError: 'buffer' is an invalid keyword argument for load_cert_chain()